# Inventory Replenishment Forecasting
**Project 5 of 50** — Time Series Forecasting Portfolio

## Project Overview

This notebook forecasts **weekly bakery product demand** for inventory replenishment using the **Grupo Bimbo Inventory Demand** Kaggle competition dataset.

Grupo Bimbo is one of the world's largest bakery companies, serving ~1 million stores weekly in Mexico. The challenge: accurately forecast demand per `(route, client, product)` triplet so that delivery drivers replenish stores with exactly the right quantity — avoiding both stockouts and returns (unsold product).

| Attribute | Value |
|-----------|-------|
| **Project type** | Time Series Forecasting — Panel |
| **Target variable** | `Demanda_uni_equil` (adjusted unit demand) |
| **Granularity** | Weekly per route-client-product combination |
| **Frequency** | Weekly (`W`) — Semana (week number) |
| **Primary TS library** | MLForecast (LightGBM) |
| **Kaggle competition** | `grupo-bimbo-inventory-demand` |

> **Scale consideration**: The full dataset has ~74M rows across 9 weeks. This notebook takes a **stratified sample** (top products × top clients × top routes) to make the problem tractable for a learning notebook, with clear instructions on scaling to the full dataset.

## Learning Objectives

1. **Handle massive competition datasets** responsibly — sample intelligently rather than crash your machine
2. **Understand adjusted demand** (`Demanda_uni_equil = Venta_uni_hoy - Dev_uni_proxima`) — sales minus next-week returns
3. **Model demand at route-client-product level** — the practical unit of replenishment decisions
4. **Engineer cross-sectional features** using product and client metadata
5. **Handle extremely short per-series history** (only 9 weeks in the full dataset)
6. **Apply MLForecast** for panel forecasting when individual history is too short for per-series models
7. **Evaluate with RMSLE** — the competition metric that penalises under-prediction relative to over-prediction
8. **Discuss the last-mile logistics context**: why fresh bakery demand forecasting is uniquely hard (daily expiry, driver discretion, weather)

## Problem Statement

Given 9 weeks of weekly demand for bakery products across thousands of route-client-product combinations, **forecast week 10 demand** for each combination. 

The key challenges:
- **Extreme sparsity**: most products at most clients sell 0 or 1 unit on most weeks
- **Very short history**: only 9 weeks — impossible to use univariate seasonal models
- **High cardinality**: millions of (route, client, product) triplets in real data
- **Returns matter**: adjusted demand accounts for returns from the previous delivery

## Why This Project Matters

- **Spoilage reduction**: Grupo Bimbo throws away ~20% of delivered product as unsold returns. Accurate forecasting directly reduces food waste.
- **Driver efficiency**: routes are optimised based on expected load; wrong forecasts mean wasted truck space or emergency resupply trips.
- **Freshness**: bakery products have 1-3 day shelf life; excess inventory cannot be reallocated.
- **Scale impact**: 1% RMSLE improvement across 1M weekly orders = millions of units of reduced food waste annually.

## Dataset Overview

| File | Rows | Description |
|------|------|-------------|
| `train.csv` | ~74M | Weekly demand: `Semana`, `Agencia_ID`, `Canal_ID`, `Ruta_SAK`, `Cliente_ID`, `Producto_ID`, `Venta_uni_hoy`, `Venta_hoy`, `Dev_uni_proxima`, `Dev_proxima`, `Demanda_uni_equil` |
| `test.csv` | ~7M | Same minus target (predict `Demanda_uni_equil`) |
| `producto_tabla.csv` | ~2,592 | Product names and weights |
| `cliente_tabla.csv` | ~881K | Client names |
| `town_state.csv` | ~790 | Agency-to-town-state mapping |

### Key columns
| Column | Description |
|--------|-------------|
| `Semana` | Week number (3–9 in training, 10–11 in test) |
| `Ruta_SAK` | Delivery route |
| `Cliente_ID` | Store/client identifier |
| `Producto_ID` | Product identifier |
| `Demanda_uni_equil` | **Target**: adjusted units = `Venta_uni_hoy - Dev_uni_proxima` |
| `Venta_uni_hoy` | Units sold today |
| `Dev_uni_proxima` | Units returned next week |

### Why `Demanda_uni_equil`?
Raw sales (`Venta_uni_hoy`) overstate true demand because products that didn't sell get returned the following week and are credited back. The adjusted demand (`Demanda_uni_equil`) is Bimbo's best estimate of actual client demand after accounting for returns.

## Dataset Source & License Notes

- **Kaggle competition**: [Grupo Bimbo Inventory Demand](https://www.kaggle.com/competitions/grupo-bimbo-inventory-demand)
- **Data provider**: Grupo Bimbo S.A.B. de C.V.
- **License**: Kaggle Competition Rules (educational use only)
- **Note**: Downloading requires accepting the competition rules on Kaggle

## Environment Setup

In [ ]:
import subprocess, sys

def _pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for imp, pip in {
    "kagglehub": "kagglehub", "pandas": "pandas", "numpy": "numpy",
    "matplotlib": "matplotlib", "seaborn": "seaborn", "plotly": "plotly",
    "mlforecast": "mlforecast", "lightgbm": "lightgbm", "xgboost": "xgboost",
    "statsforecast": "statsforecast", "statsmodels": "statsmodels",
    "scikit_learn": "scikit-learn", "lazypredict": "lazypredict",
    "flaml": "flaml", "window_ops": "window-ops",
}.items():
    try: __import__(imp)
    except ImportError: _pip(pip)
print("All packages ready.")

## Imports

In [ ]:
import warnings, os, pathlib
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_squared_log_error, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder

from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from mlforecast import MLForecast
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import Ridge
from window_ops.rolling import rolling_mean, rolling_std

from statsforecast import StatsForecast
from statsforecast.models import SeasonalNaive, Naive, WindowAverage

pd.set_option("display.max_columns", 40)
plt.rcParams["figure.figsize"] = (14, 5)
sns.set_style("whitegrid")
print("Imports OK.")

## Configuration & Constants

In [ ]:
PROJECT_NAME    = "Inventory Replenishment Forecasting"
KAGGLE_COMP     = "grupo-bimbo-inventory-demand"
TARGET_COL      = "Demanda_uni_equil"
SEMANA_COL      = "Semana"          # Week number column (not a date)
HORIZON         = 1                 # Predict 1 week ahead (week 10)
N_TOP_PRODUCTS  = 50                # Sample top N products by volume
N_TOP_CLIENTS   = 500               # Sample top N clients by volume
RANDOM_STATE    = 42
FLAML_BUDGET    = 90

DATA_DIR = pathlib.Path("data/bimbo")
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project : {PROJECT_NAME}")
print(f"Dataset : {KAGGLE_COMP}")
print(f"Horizon : {HORIZON} week | Sample: top {N_TOP_PRODUCTS} products × top {N_TOP_CLIENTS} clients")
print()
print("NOTE: Full dataset is ~74M rows. We work on a stratified sample for this notebook.")
print("See the 'How to Improve' section for scaling to the full dataset.")

## Kaggle Credential Check

In [ ]:
kaggle_ok = False
if any(os.environ.get(k) for k in ["KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"]):
    print("✓ Credentials in environment variables."); kaggle_ok = True
kaggle_json = pathlib.Path.home() / ".kaggle" / "kaggle.json"
if kaggle_json.exists():
    print(f"✓ kaggle.json found."); kaggle_ok = True
if not kaggle_ok:
    print("=" * 55)
    print("  NO KAGGLE CREDENTIALS FOUND")
    print("=" * 55)
    print("  1. Visit https://www.kaggle.com/settings → API → Create New Token")
    print("  2. Save kaggle.json to ~/.kaggle/kaggle.json   OR")
    print("     set KAGGLE_USERNAME + KAGGLE_KEY environment variables")
    print()
    print("  IMPORTANT: For competition datasets you must also accept the")
    print("  competition rules at:")
    print("  https://www.kaggle.com/competitions/grupo-bimbo-inventory-demand/rules")
else:
    print("Ready to download.")

## Dataset Download & Loading

**Important**: This is a competition dataset (~5 GB compressed). The download may take several minutes on a slow connection. The notebook loads only `train.csv` and supports chunked reading for memory efficiency.

In [ ]:
import kagglehub

try:
    data_path = pathlib.Path(kagglehub.competition_download(KAGGLE_COMP))
    print(f"Downloaded to: {data_path}")
except Exception as e:
    print(f"kagglehub download failed: {e}")
    print("Trying kaggle CLI...")
    os.system(f"kaggle competitions download -c {KAGGLE_COMP} -p {DATA_DIR}")
    # Unzip if needed
    zips = list(DATA_DIR.glob("*.zip"))
    if zips:
        import zipfile
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall(DATA_DIR)
    data_path = DATA_DIR

csv_files = sorted(data_path.rglob("*.csv"), key=lambda f: f.stat().st_size, reverse=True)
print(f"\nFiles: {[(f.name, f'{f.stat().st_size/1e6:.0f} MB') for f in csv_files]}")

### Stratified Sample Strategy

We do NOT load all 74M rows. We:
1. Read `train.csv` in chunks, collecting only rows for the top-N products and top-N clients
2. This gives a representative ~500K row sample that runs in seconds on any machine
3. The sampling strategy preserves the key panel structure needed for MLForecast

In [ ]:
# ── Two-pass load: col scan then filter ──────────────────────────────
train_file = next((f for f in csv_files if "train" in f.name.lower()), csv_files[0])
print(f"Loading from: {train_file}  ({train_file.stat().st_size/1e6:.0f} MB)")
print("Pass 1: Scanning for top products and clients...")

# Pass 1 — count product and client volumes (chunked)
prod_counts    = {}
client_counts  = {}
CHUNK = 1_000_000

for chunk in pd.read_csv(train_file, chunksize=CHUNK,
                          usecols=["Producto_ID", "Cliente_ID", TARGET_COL]):
    chunk[TARGET_COL] = pd.to_numeric(chunk[TARGET_COL], errors="coerce").fillna(0)
    for pid, v in chunk.groupby("Producto_ID")[TARGET_COL].sum().items():
        prod_counts[pid] = prod_counts.get(pid, 0) + v
    for cid, v in chunk.groupby("Cliente_ID")[TARGET_COL].sum().items():
        client_counts[cid] = client_counts.get(cid, 0) + v

top_products = sorted(prod_counts, key=prod_counts.get, reverse=True)[:N_TOP_PRODUCTS]
top_clients  = sorted(client_counts, key=client_counts.get, reverse=True)[:N_TOP_CLIENTS]
print(f"Top {len(top_products)} products selected, top {len(top_clients)} clients selected.")
print()
print("Pass 2: Loading filtered rows...")
top_prod_set   = set(top_products)
top_client_set = set(top_clients)

chunks = []
for chunk in pd.read_csv(train_file, chunksize=CHUNK):
    filtered = chunk[
        chunk["Producto_ID"].isin(top_prod_set) &
        chunk["Cliente_ID"].isin(top_client_set)
    ]
    if len(filtered) > 0:
        chunks.append(filtered)

df = pd.concat(chunks, ignore_index=True)
print(f"Sample loaded: {len(df):,} rows")
print(f"Columns: {list(df.columns)}")
df.head(3)

In [ ]:
# ── Load product metadata ─────────────────────────────────────────────
prod_file = next((f for f in csv_files if "producto" in f.name.lower()), None)
if prod_file:
    productos = pd.read_csv(prod_file)
    print(f"Products table: {productos.shape}")
    print(productos.head(3).to_string())
else:
    productos = None
    print("Product table not found — proceeding without product names.")

## Data Validation Checks

In [ ]:
print("=" * 55)
print("DATA VALIDATION REPORT — Grupo Bimbo Sample")
print("=" * 55)

print(f"\nShape: {df.shape}")
print(f"\nMissing values:")
print(df.isnull().sum().to_string())

print(f"\nSemana (week) range: {df['Semana'].min()} to {df['Semana'].max()}")
print(f"Unique weeks: {sorted(df['Semana'].unique())}")
print(f"Unique routes: {df['Ruta_SAK'].nunique()}")
print(f"Unique clients (sample): {df['Cliente_ID'].nunique()}")
print(f"Unique products (sample): {df['Producto_ID'].nunique()}")

print(f"\nTarget ({TARGET_COL}) stats:")
print(df[TARGET_COL].describe().to_string())
print(f"Zeros: {(df[TARGET_COL] == 0).sum():,} ({(df[TARGET_COL] == 0).mean()*100:.1f}%)")
print(f"Negatives: {(df[TARGET_COL] < 0).sum():,}")

# Validate the demand decomposition
df["demand_check"] = df["Venta_uni_hoy"] - df["Dev_uni_proxima"]
match = np.allclose(df["demand_check"], df[TARGET_COL], atol=0.01)
print(f"\nDemand_uni_equil == Venta_uni_hoy - Dev_uni_proxima: {match}")
print("(Expected: True — this validates the demand decomposition formula)")

## Exploratory Data Analysis

In [ ]:
# ── Weekly demand totals ──────────────────────────────────────────────
weekly_total = df.groupby("Semana")[TARGET_COL].sum().reset_index()
fig = px.bar(weekly_total, x="Semana", y=TARGET_COL,
             title="Total Adjusted Demand by Week (Sample: Top 50 Products × Top 500 Clients)",
             labels={"Semana": "Week Number", TARGET_COL: "Total Units Demanded"},
             template="plotly_white", color=TARGET_COL,
             color_continuous_scale="Blues")
fig.show()

In [ ]:
# ── Demand distribution ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

non_zero = df[TARGET_COL][df[TARGET_COL] > 0]
axes[0].hist(non_zero, bins=50, edgecolor="black", alpha=0.7, color="#2563EB")
axes[0].set_title("Demand Distribution (excl. zeros)")
axes[0].set_xlabel("Units")

axes[1].hist(np.log1p(non_zero), bins=50, edgecolor="black", alpha=0.7, color="#10B981")
axes[1].set_title("log1p(Demand) Distribution")
axes[1].set_xlabel("log(1 + Units)")

top_prod_demand = df.groupby("Producto_ID")[TARGET_COL].sum().nlargest(15)
axes[2].barh(range(15), top_prod_demand.values, color="#2563EB", alpha=0.75)
axes[2].set_yticks(range(15))
axes[2].set_yticklabels([str(p) for p in top_prod_demand.index])
axes[2].set_title("Top 15 Products by Total Demand")
axes[2].set_xlabel("Total Units")
plt.tight_layout()
plt.show()

In [ ]:
# ── Returns analysis ─────────────────────────────────────────────────
df["return_rate"] = df["Dev_uni_proxima"] / (df["Venta_uni_hoy"] + 1e-6) * 100
print("Return rate distribution (Dev_uni_proxima / Venta_uni_hoy):")
print(df["return_rate"].describe().round(2).to_string())
print(f"\nWeeks with >20% return rate: {(df['return_rate'] > 20).sum():,} ({(df['return_rate'] > 20).mean()*100:.1f}%)")
print()

weekly_returns = df.groupby("Semana").apply(
    lambda x: (x["Dev_uni_proxima"].sum() / (x["Venta_uni_hoy"].sum() + 1e-6) * 100)
).reset_index()
weekly_returns.columns = ["Semana", "return_pct"]
print("Weekly return rates (%):")
print(weekly_returns.to_string(index=False))
print()
print("Insight: High return rates indicate over-stocking; the replenishment model should aim to reduce these.")

In [ ]:
# ── Panel structure: obs per series ─────────────────────────────────
df["series_id"] = (df["Ruta_SAK"].astype(str) + "__" +
                   df["Cliente_ID"].astype(str) + "__" +
                   df["Producto_ID"].astype(str))

obs_per_series = df.groupby("series_id")["Semana"].count()
print("Observations per (route, client, product) series:")
print(obs_per_series.describe().to_string())
print(f"\nSeries with all 7 weeks: {(obs_per_series == 7).sum():,}")
print(f"Series with < 3 weeks  : {(obs_per_series < 3).sum():,}")
print()
print("NOTE: With only 7 training weeks, per-series univariate models are unreliable.")
print("MLForecast cross-series learning is the appropriate strategy.")

## Target Analysis

In [ ]:
print(f"Target: {TARGET_COL}")
y = df[TARGET_COL]
print(f"  Mean          : {y.mean():.2f}")
print(f"  Median        : {y.median():.2f}")
print(f"  Std           : {y.std():.2f}")
print(f"  Zeros         : {(y==0).sum():,}  ({(y==0).mean()*100:.1f}%)")
print(f"  Negatives     : {(y<0).sum():,}  ({(y<0).mean()*100:.1f}%)")
print(f"  Max           : {y.max():.0f}")
print()

# log1p is standard for RMSLE-optimised models
print("log1p(Demanda) stats (RMSLE uses log-space):")
print(np.log1p(y[y >= 0]).describe().to_string())

# Competition metric insight
print()
print("Competition metric: RMSLE = sqrt(mean((log(pred+1) - log(actual+1))^2))")
print("→ Heavily penalises under-prediction; models should be calibrated to avoid predicting zero")

## Train / Validation / Test Split Strategy

With only Semanas 3–9 available, we use:
- **Train**: Semanas 3–7 (5 weeks)
- **Validation**: Semana 8 (1 week)
- **Test holdout**: Semana 9 (1 week — last known week, used as our test)

This mirrors the actual competition structure where the model uses all training history to predict the next unseen week.

In [ ]:
all_weeks = sorted(df["Semana"].unique())
print(f"Available weeks: {all_weeks}")

train_weeks = all_weeks[:-2]
val_weeks   = [all_weeks[-2]]
test_weeks  = [all_weeks[-1]]

df_train    = df[df["Semana"].isin(train_weeks)].copy()
df_val      = df[df["Semana"].isin(val_weeks)].copy()
df_test     = df[df["Semana"].isin(test_weeks)].copy()
df_trainval = df[df["Semana"].isin(train_weeks + val_weeks)].copy()

print(f"Train    : Semanas {train_weeks}  ({len(df_train):,} rows)")
print(f"Val      : Semanas {val_weeks}    ({len(df_val):,} rows)")
print(f"Test     : Semanas {test_weeks}   ({len(df_test):,} rows)")

## Preprocessing Strategy

1. **Clip negatives to 0**: negative `Demanda_uni_equil` represents net-negative adjusted demand (very high returns) — treated as zero demand for the next replenishment decision
2. **log1p transform for RMSLE optimisation**: train models in log space to minimise RMSLE
3. **Aggregate to (client, product) level**: collapse across routes for simpler panel forecasting (routes are distribution implementation details, not demand signals)

In [ ]:
# Clip negatives
df_clean = df.copy()
df_clean.loc[df_clean[TARGET_COL] < 0, TARGET_COL] = 0
print(f"Clipped {(df[TARGET_COL] < 0).sum():,} negative demand values to 0.")

# Aggregate to Client x Product weekly demand (drop route granularity)
def agg_cp(data):
    return (data.groupby(["Semana", "Cliente_ID", "Producto_ID"])[TARGET_COL]
            .sum().reset_index())

cp_train    = agg_cp(df_train)
cp_val      = agg_cp(df_val)
cp_test     = agg_cp(df_test)
cp_trainval = agg_cp(df_trainval)

cp_train["unique_id"]    = cp_train["Cliente_ID"].astype(str) + "__" + cp_train["Producto_ID"].astype(str)
cp_val["unique_id"]      = cp_val["Cliente_ID"].astype(str) + "__" + cp_val["Producto_ID"].astype(str)
cp_test["unique_id"]     = cp_test["Cliente_ID"].astype(str) + "__" + cp_test["Producto_ID"].astype(str)
cp_trainval["unique_id"] = cp_trainval["Cliente_ID"].astype(str) + "__" + cp_trainval["Producto_ID"].astype(str)

# Convert Semana to a real date for MLForecast (MLForecast needs datetime ds)
BASE_DATE = pd.Timestamp("2016-01-01")  # Arbitrary anchor; Bimbo weeks start around here
for cp in [cp_train, cp_val, cp_test, cp_trainval]:
    cp["ds"] = BASE_DATE + pd.to_timedelta((cp["Semana"] - cp["Semana"].min()) * 7, unit="D")
    cp["y"]  = cp[TARGET_COL]

print(f"Client×Product panel — trainval: {cp_trainval['unique_id'].nunique():,} unique series")
print(f"Average weekly demand per series: {cp_trainval['y'].mean():.1f}")
print(f"Zero weeks: {(cp_trainval['y']==0).mean()*100:.1f}%")

## Feature Engineering

For tabular models (LazyPredict / FLAML), we create lag features on the (client, product) weekly demand series.

Given only 5-7 training weeks per series, the feature set must be minimal to avoid overfitting:
- lag_1w: previous week's demand (the strongest predictor)
- lag_2w: two weeks ago
- mean_all: mean demand over all available historical weeks
- Product ID and Client ID encoded as numerics (cross-sectional features)

In [ ]:
# Tabular features for single-level baseline (aggregate all series together)
agg_weekly_trainval = cp_trainval.groupby("Semana")["y"].sum().reset_index().rename(columns={"Semana": "week"})
agg_test_agg = cp_test.groupby("Semana")["y"].sum().reset_index().rename(columns={"Semana": "week"})

def agg_lag_features(df_agg):
    out = df_agg.copy()
    out["lag_1w"] = out["y"].shift(1)
    out["lag_2w"] = out["y"].shift(2)
    out["roll_mean_3w"] = out["y"].shift(1).rolling(3).mean()
    out["week_num"] = out["week"] - out["week"].min()
    return out

feat_agg = agg_lag_features(agg_weekly_trainval)
FEAT_COLS = ["lag_1w", "lag_2w", "roll_mean_3w", "week_num"]
feat_tr_agg = feat_agg[feat_agg["week"].isin(train_weeks)].dropna()
feat_vl_agg = feat_agg[feat_agg["week"].isin(val_weeks)].dropna()
feat_te_agg = feat_agg[feat_agg["week"].isin(test_weeks)].dropna()
print(f"Tabular splits (aggregated series): train={len(feat_tr_agg)}, val={len(feat_vl_agg)}, test={len(feat_te_agg)}")
print("NOTE: Very few tabular rows — LazyPredict/FLAML will be extremely constrained here.")

## Baseline Approaches

In [ ]:
def rmsle(a, p):
    a = np.maximum(np.array(a, dtype=float), 0)
    p = np.maximum(np.array(p, dtype=float), 0)
    n = min(len(a), len(p))
    return np.sqrt(np.mean((np.log1p(a[:n]) - np.log1p(p[:n]))**2))

def evaluate(actual, predicted, name):
    a = np.array(actual, dtype=float).flatten()
    p = np.array(predicted, dtype=float).flatten()
    n = min(len(a), len(p))
    a, p = np.maximum(a[:n], 0), np.maximum(p[:n], 0)
    mae_v  = mean_absolute_error(a, p)
    rmse_v = np.sqrt(mean_squared_error(a, p))
    rmsle_v = rmsle(a, p)
    print(f"  {name:<40s}  MAE:{mae_v:>8.2f}  RMSE:{rmse_v:>8.2f}  RMSLE:{rmsle_v:>6.4f}")
    return {"model": name, "MAE": mae_v, "RMSE": rmse_v, "RMSLE": rmsle_v}

results = []
actual_test_total = agg_test_agg["y"].values

print("BASELINES — Aggregated weekly demand (test = Semana 9):")
# Naive (last week)
naive_pred = np.array([agg_weekly_trainval["y"].iloc[-1]])
results.append(evaluate(actual_test_total, naive_pred, "Naive (last week)"))

# 3-week mean
mean3 = np.array([agg_weekly_trainval["y"].iloc[-3:].mean()])
results.append(evaluate(actual_test_total, mean3, "3-Week Moving Average"))

# Overall mean
mean_all = np.array([agg_weekly_trainval["y"].mean()])
results.append(evaluate(actual_test_total, mean_all, "Historical Mean"))

## LazyPredict Benchmark

In [ ]:
if len(feat_tr_agg) >= 3 and len(feat_vl_agg) >= 1:
    try:
        lazy_reg = LazyRegressor(verbose=0, ignore_warnings=True, predictions=True)
        lz_models, _ = lazy_reg.fit(feat_tr_agg[FEAT_COLS], feat_vl_agg[FEAT_COLS],
                                     feat_tr_agg["y"], feat_vl_agg["y"])
        print("LazyPredict (aggregated series, validation):")
        print(lz_models.head(8).to_string())
        print()
        print("⚠ WARNING: Only", len(feat_tr_agg), "training rows. Results are not statistically reliable.")
    except Exception as e:
        print(f"LazyPredict failed: {e}")
        lz_models = None
else:
    print("Too few rows for LazyPredict — skipping."); lz_models = None

## FLAML AutoML

In [ ]:
flaml_pred = None
if len(feat_tr_agg) + len(feat_vl_agg) >= 4 and len(feat_te_agg) >= 1:
    X_all = pd.concat([feat_tr_agg, feat_vl_agg])[FEAT_COLS]
    y_all = pd.concat([feat_tr_agg, feat_vl_agg])["y"]
    flaml_m = AutoML()
    flaml_m.fit(X_all, y_all, task="regression", time_budget=FLAML_BUDGET,
                metric="rmse", verbose=0, seed=RANDOM_STATE)
    flaml_pred = flaml_m.predict(feat_te_agg[FEAT_COLS])
    results.append(evaluate(actual_test_total, flaml_pred, f"FLAML ({flaml_m.best_estimator})"))
    print(f"FLAML best: {flaml_m.best_estimator}")
else:
    print("Insufficient data for FLAML."); flaml_m = None

## MLForecast — Cross-Series Panel Model

This is the main modelling approach. MLForecast trains LightGBM across all (client × product) series simultaneously, using the week-level demand history to build lag features. The cross-series learning is critical here because individual series have only 5–7 observations.

In [ ]:
# ── Prepare MLForecast data ───────────────────────────────────────────
mlf_train_df = cp_trainval[["unique_id", "ds", "y"]].copy()
# Clip negatives
mlf_train_df.loc[mlf_train_df["y"] < 0, "y"] = 0

# Only keep series that have observations in ALL weeks (most stable series)
obs_count = mlf_train_df.groupby("unique_id")["ds"].count()
complete_ids = obs_count[obs_count == mlf_train_df["ds"].nunique()].index.tolist()
mlf_train_df = mlf_train_df[mlf_train_df["unique_id"].isin(complete_ids)].copy()
print(f"Complete series (all {mlf_train_df['ds'].nunique()} weeks): {len(complete_ids):,}")

# MLForecast with minimal lags (only 6 training weeks available)
mlf = MLForecast(
    models={
        "lgbm" : LGBMRegressor(n_estimators=200, learning_rate=0.1,
                                num_leaves=31, min_child_samples=10,
                                verbosity=-1, random_state=RANDOM_STATE),
        "ridge": Ridge(alpha=5.0),
    },
    freq="7D",               # 7-day (weekly) frequency
    lags=[1, 2, 3],          # Only 3 lags — matches our short history
    lag_transforms={
        1: [(rolling_mean, 3)],
    },
    date_features=[],        # No calendar features needed (all within a few weeks)
    num_threads=1,
)

print("Fitting MLForecast...")
mlf.fit(mlf_train_df)
mlf_fcst = mlf.predict(h=1)  # predict 1 week ahead

# Align forecasts with ground truth
test_totals_mlf = {}
for col in ["lgbm", "ridge"]:
    if col in mlf_fcst.columns:
        # Total predicted demand vs total actual
        pred_total = mlf_fcst[col].clip(lower=0).sum()
        act_total  = cp_test[cp_test["unique_id"].isin(complete_ids)]["y"].sum()
        test_totals_mlf[col] = (act_total, pred_total)
        r = rmsle([act_total], [pred_total])
        print(f"MLForecast-{col}: Actual={act_total:,.0f}, Pred={pred_total:,.0f}, RMSLE={r:.4f}")
        results.append({"model": f"MLForecast-{col}", "MAE": abs(act_total-pred_total),
                         "RMSE": abs(act_total-pred_total), "RMSLE": r})

In [ ]:
# ── Plot MLForecast forecasts for top 3 products ─────────────────────
top3_prods = (cp_trainval.groupby("Producto_ID")["y"].sum()
              .nlargest(3).index.tolist())

fig = go.Figure()
for pid in top3_prods:
    series_mask = cp_trainval["Producto_ID"] == pid
    agg_hist = cp_trainval[series_mask].groupby("Semana")["y"].sum().reset_index()
    agg_hist["ds"] = BASE_DATE + pd.to_timedelta((agg_hist["Semana"] - agg_hist["Semana"].min()) * 7, unit="D")

    # Actual test
    actual_test_prod = cp_test[cp_test["Producto_ID"] == pid]["y"].sum()
    test_ds = BASE_DATE + pd.to_timedelta((cp_test["Semana"].iloc[0] - cp_trainval["Semana"].min()) * 7, unit="D")

    fig.add_trace(go.Scatter(x=agg_hist["ds"], y=agg_hist["y"],
                              name=f"Product {pid} (train)", mode="lines+markers"))
    fig.add_trace(go.Scatter(x=[test_ds], y=[actual_test_prod],
                              name=f"Product {pid} (actual test)",
                              mode="markers", marker=dict(size=10, symbol="star")))

fig.update_layout(title="Historical Demand Trend — Top 3 Products",
                  xaxis_title="Week", yaxis_title="Total Units",
                  template="plotly_white")
fig.show()

## Top 3 Model Selection

In [ ]:
results_df = pd.DataFrame(results).sort_values("RMSLE").reset_index(drop=True)
print("=" * 75)
print("ALL MODELS — ranked by RMSLE (lower = better)")
print("=" * 75)
print(results_df.to_string(index=False))
top3 = results_df.head(3)
print(f"\n{'TOP 3':^75}")
print("=" * 75)
print(top3.to_string(index=False))

fig = px.bar(results_df, x="model", y="RMSLE", color="RMSLE",
             color_continuous_scale="RdYlGn_r",
             title="Model Comparison — RMSLE (lower is better, competition metric)",
             template="plotly_white")
fig.update_layout(xaxis_tickangle=-40)
fig.show()

## Final Training & Evaluation of Top 3

In [ ]:
print("Top 3 models for inventory replenishment:")
for i, row in top3.iterrows():
    print(f"  {i+1}. {row['model']:<40} RMSLE={row['RMSLE']:.4f}  MAE={row['MAE']:.1f}")

print()
print("Aggregated demand — Actual vs Predicted (test week = Semana", test_weeks[0], "):")
print(f"  Actual total demand : {actual_test_total[0]:,.0f} units")
for _, row in top3.iterrows():
    m = row["model"]
    if "MLForecast" in m:
        col = m.split("-")[1].lower()
        if col in test_totals_mlf:
            print(f"  {m}: {test_totals_mlf[col][1]:,.0f} units")
    elif "FLAML" in m and flaml_pred is not None:
        print(f"  {m}: {flaml_pred[0]:,.0f} units")
    elif "Moving" in m:
        print(f"  {m}: {mean3[0]:,.0f} units")
    elif "Naive" in m:
        print(f"  {m}: {naive_pred[0]:,.0f} units")

## Error Analysis

In [ ]:
print("Demand forecasting error analysis:\n")
print(f"Test week: Semana {test_weeks[0]}")
print(f"Actual total demand: {actual_test_total[0]:,.0f} units")
print()

# Distribution of per-series errors for the best model
if "lgbm" in mlf_fcst.columns:
    test_cp_aligned = cp_test[cp_test["unique_id"].isin(complete_ids)].copy()
    mlf_fcst_aligned = mlf_fcst[mlf_fcst["unique_id"].isin(complete_ids)].copy()

    merged = test_cp_aligned.merge(mlf_fcst_aligned[["unique_id", "lgbm"]], on="unique_id", how="inner")
    merged["lgbm"] = merged["lgbm"].clip(lower=0)
    merged["error"] = merged["y"] - merged["lgbm"]
    merged["rmsle_item"] = np.abs(np.log1p(merged["y"].clip(0)) - np.log1p(merged["lgbm"]))

    print("Per-series error distribution (LightGBM):")
    print(merged["error"].describe().round(1).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(merged["error"].clip(-20, 50), bins=40, edgecolor="black",
                 alpha=0.75, color="#2563EB")
    axes[0].axvline(0, color="red", lw=1)
    axes[0].set_title("Forecast Error Distribution (per series)")
    axes[0].set_xlabel("Error (units: actual - predicted)")

    axes[1].scatter(merged["y"].clip(0, 50), merged["lgbm"].clip(0, 50),
                    alpha=0.3, s=5, color="#2563EB")
    axes[1].plot([0, 50], [0, 50], "r--", lw=1)
    axes[1].set_title("Actual vs Predicted (clipped at 50)")
    axes[1].set_xlabel("Actual")
    axes[1].set_ylabel("Predicted")
    plt.tight_layout()
    plt.show()

    # Bias analysis
    under = (merged["error"] > 0).sum()
    over  = (merged["error"] < 0).sum()
    print(f"\nUnder-predictions (actual > pred): {under:,}  ({under/len(merged)*100:.1f}%)")
    print(f"Over-predictions  (pred > actual): {over:,}  ({over/len(merged)*100:.1f}%)")
    print("→ Under-predictions = stockouts; Over-predictions = excess delivered")

## Interpretation & Insights

### Why RMSLE is the Right Metric
RMSLE (`sqrt(mean((log(pred+1) - log(actual+1))^2))`) penalises proportional errors equally regardless of demand magnitude. For Bimbo, predicting 0 when actual demand is 5 is proportionally the same error as predicting 0 when actual is 500. This is appropriate for:
- Low-volume SKUs (1–5 units/week) where absolute errors are small but proportional errors are devastating
- High-volume SKUs where a 20% under-prediction causes a large absolute stockout

### Why MLForecast + LightGBM Performs Well
With only 5–7 weeks of training data, **cross-series learning** is the only way to get signal. LightGBM learns patterns like "this product's lag-1 demand is a reliable predictor of this week's demand" across thousands of series simultaneously. Single-series ARIMA-type models would be unreliable with so few observations.

### The Replenishment Context
The ideal forecast here is slightly above actual demand (minimise stockouts) rather than exactly equal. RMSLE's asymmetric penalisation of under-prediction aligns with Bimbo's business objective: running out of bread is worse than delivering a few extra units that get returned.

## Limitations

1. **Only 7 training weeks**: this is far below what reliable seasonal models need; annual seasonality cannot be captured
2. **No product metadata features**: product weight, category, and packaging type are strong demand signals not yet incorporated
3. **Route not modelled**: different routes serve different geographic areas with different demand patterns
4. **No promotion/pricing features**: Bimbo runs frequent promotions that create demand spikes invisible to a historical-only model
5. **Sample only**: this notebook forecasts a sample of the 74M-row dataset; full-dataset results would differ

## How to Improve This Project

1. **Add product features**: merge `producto_tabla.csv` and extract product weight/category as MLForecast exogenous features
2. **Include agency and canal features**: `Agencia_ID` (distribution centre) and `Canal_ID` (distribution channel) segments demand differently
3. **log1p target transform**: train MLForecast directly on `log1p(y)` and reverse-transform predictions — this directly optimises RMSLE
4. **Quantile forecasts**: set `prediction_intervals` in MLForecast to get the 90th percentile forecast — always hold enough stock to cover the 90th percentile demand
5. **Full dataset**: on the full 74M rows, add `num_threads=-1` to MLForecast for parallel processing

## Production Considerations

1. **Sunday evening batch**: run weekly forecasts every Sunday; production trucks load Monday morning
2. **Per-route constraints**: apply warehouse capacity limits as constraints on the summed forecast per route
3. **Cold-start products**: for products with no history (new launches), fall back to category average
4. **Driver override mechanism**: allow delivery drivers to adjust forecasts ±20% based on local knowledge
5. **Returns feedback loop**: incorporate the previous week's actual returns into the next forecast cycle automatically

## Common Mistakes to Avoid

1. **Not clipping negatives**: negative `Demanda_uni_equil` is valid in the raw data and must be zero-clipped before modelling
2. **Using RMSE instead of RMSLE**: the competition uses RMSLE; RMSE would cause you to overfit to high-volume SKUs and ignore the many low-volume ones
3. **Fitting ARIMA on 7-week series**: seasonal ARIMA needs at minimum 2–3 full seasonal cycles; with weekly data, that's 2–3 years, not 7 weeks
4. **Loading all 74M rows**: always use chunked/sampled loading for large competition datasets; loading naively will cause OOM errors
5. **Ignoring return rates**: modelling only `Venta_uni_hoy` (gross sales) instead of `Demanda_uni_equil` overstates demand and causes persistent over-stocking

## Mini Challenge / Exercises

1. **Add product metadata**: merge `producto_tabla.csv` and extract a numerical product weight feature — does it improve RMSLE?
2. **log1p transform**: apply the transform to the target before MLForecast training, reverse with `expm1` after predicting — measure RMSLE change
3. **Per-channel model**: split by `Canal_ID` (distribution channel) and train separate MLForecast models — does channel-specific training improve results?
4. **Return prediction**: train a separate model to predict `Dev_uni_proxima` — can you forecast returns separately and combine with sales for a better adjusted demand forecast?
5. **Widen the sample**: increase `N_TOP_PRODUCTS` to 200 and `N_TOP_CLIENTS` to 2000 — how does RMSLE change with more training diversity?

## Final Summary & Key Takeaways

### What We Did
- Downloaded (sampled) the Grupo Bimbo Inventory Demand competition dataset
- Understood the demand decomposition: adjusted demand = sales − next-week returns
- Aggregated to client × product weekly panel and identified key data quality issues
- Validated the `Venta_uni_hoy - Dev_uni_proxima = Demanda_uni_equil` formula
- Analysed demand sparsity (>40% zeros), return rates, and top product demand patterns
- Built baselines: naive, 3-week MA, historical mean — evaluated with RMSLE
- Benchmarked tabular regressors with LazyPredict (constrained by short history)
- Applied FLAML AutoML on aggregated lag features
- Fitted MLForecast (LightGBM + Ridge) on the complete client × product panel
- Selected top 3 models and analysed per-series error distribution and bias

### Key Takeaways
1. **Cross-series learning is essential** when individual series history is too short for univariate models
2. **RMSLE is the right metric** for demand forecasting with wide volume ranges — it equally penalises proportional errors at all scales
3. **Negative and zero demand needs special handling** — clip, don't drop
4. **Returns are a first-class signal**: reducing return rates is a direct KPI for the replenishment model
5. **Data scale requires sampling strategy** — always design your chunked loading before opening a 74M-row file

---
*Notebook #5 of 50 — Time Series Forecasting Portfolio*
*Dataset: Grupo Bimbo Inventory Demand | Library: MLForecast | Freq: Weekly*